## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Capability contract

**Capability:** C03 — Correctness Contracts & Idempotency  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Detect a correctness failure


## 🖊️ Sketch note

![Sketch note — Incremental & Idempotent](../assets/sketch-notes/13-incremental-idempotent.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Your task

**Goal:** Make incremental loading observable and safe to retry.

**Do this:** Add a run audit containing run ID, source window, input/inserted/duplicate rows, final watermark and status. Simulate a failed run and show how an operator decides retry vs backfill.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S04 — Incremental and idempotent pipelines (DATA-1070)** → [`../work_simulations/04_acme_commerce_duplicate_rerun/README.md`](../work_simulations/04_acme_commerce_duplicate_rerun/README.md)
- **S06 — Retries and backfills (DATA-1094)** → [`../work_simulations/06_acme_commerce_retry_side_effect/README.md`](../work_simulations/06_acme_commerce_retry_side_effect/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Measure

Record an observable result that lets you compare your prediction with what actually happened.


## Learning objectives

- explain incremental state;
- reproduce duplicate processing;
- make a load idempotent;
- reason about late-arriving data.


## 1. Mental model

```text
source events → select a window → deduplicate → upsert → advance checkpoint
                         ↑                         ↓
                    late data                 durable state
```

The key idea: **progress state and data state must be consistent**. Advancing a watermark before the data is safely committed can lose records; writing data without a stable key can create duplicates.

## 2. Predict before you run

Suppose a batch contains events 1–3 and the worker crashes after writing them but before recording the checkpoint.

> What happens when the same batch is retried? What must the destination guarantee?

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"duckdb": "duckdb>=1.0,<2", "pyarrow": "pyarrow>=16,<22", "pandas": "pandas>=2,<4", "requests": "requests>=2.31,<3"}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import duckdb

WORK = Path("data/13_incremental")
WORK.mkdir(parents=True, exist_ok=True)
DB = WORK / "pipeline.duckdb"
if DB.exists(): DB.unlink()
con = duckdb.connect(str(DB))

base = datetime(2026, 9, 1, tzinfo=timezone.utc)
batch_1 = pd.DataFrame([
    {"event_id": "E1", "event_time": base + timedelta(minutes=1), "customer_id": "C1", "amount": 10},
    {"event_id": "E2", "event_time": base + timedelta(minutes=2), "customer_id": "C2", "amount": 20},
    {"event_id": "E3", "event_time": base + timedelta(minutes=3), "customer_id": "C1", "amount": 30},
])
batch_2 = pd.DataFrame([
    {"event_id": "E3", "event_time": base + timedelta(minutes=3), "customer_id": "C1", "amount": 30},  # replay
    {"event_id": "E4", "event_time": base + timedelta(minutes=4), "customer_id": "C3", "amount": 40},
])
late_batch = pd.DataFrame([
    {"event_id": "E5", "event_time": base + timedelta(minutes=2, seconds=30), "customer_id": "C2", "amount": 25},
])

con.execute("""
CREATE TABLE events (
    event_id VARCHAR PRIMARY KEY,
    event_time TIMESTAMPTZ,
    customer_id VARCHAR,
    amount DOUBLE
)
""")
con.execute("CREATE TABLE checkpoint (name VARCHAR PRIMARY KEY, watermark TIMESTAMPTZ)")
con.execute("INSERT INTO checkpoint VALUES ('events', NULL)")
print("Destination and checkpoint initialized.")

## 3. See the duplicate problem first

A naive append treats every delivery as a new event.

In [ ]:
naive = pd.concat([batch_1, batch_2], ignore_index=True)
print(naive)
print("Rows delivered:", len(naive))
print("Unique event IDs:", naive["event_id"].nunique())
print("Duplicate deliveries:", int(naive.duplicated("event_id").sum()))

## 4. Build an idempotent load

Use the event's stable identity as the destination key. The load becomes safe to retry because the same event cannot be inserted twice.

In [ ]:
def load_idempotent(df: pd.DataFrame) -> int:
    con.register("incoming", df)
    before = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
    con.execute("""
        INSERT INTO events
        SELECT i.event_id, i.event_time, i.customer_id, i.amount
        FROM incoming i
        WHERE NOT EXISTS (
            SELECT 1 FROM events e WHERE e.event_id = i.event_id
        )
    """)
    after = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
    con.unregister("incoming")
    return after - before

inserted_1 = load_idempotent(batch_1)
inserted_2 = load_idempotent(batch_2)
print("Inserted batch 1:", inserted_1)
print("Inserted replay/new batch 2:", inserted_2)
display(con.execute("SELECT * FROM events ORDER BY event_time").fetchdf())
assert inserted_1 == 3 and inserted_2 == 1
assert con.execute("SELECT COUNT(*) FROM events").fetchone()[0] == 4

## 5. Measure

Idempotency should leave measurable evidence: repeated delivery increases input rows but not destination rows.

In [ ]:
before = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
replay_rows = len(pd.concat([batch_1, batch_2], ignore_index=True))
replayed_inserted = load_idempotent(pd.concat([batch_1, batch_2], ignore_index=True))
after = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]

print("Replay input rows:", replay_rows)
print("Rows inserted from replay:", replayed_inserted)
print("Destination rows before/after:", before, after)
assert replayed_inserted == 0
assert before == after

## 5. Add a watermark

A watermark limits how much source history we inspect on a normal run. It is an optimization and coordination mechanism—not a substitute for deduplication.

In [ ]:
con.execute("UPDATE checkpoint SET watermark = ? WHERE name = 'events'", [base + timedelta(minutes=4)])
watermark = con.execute("SELECT watermark FROM checkpoint WHERE name='events'").fetchone()[0]
print("Current watermark:", watermark)

# A normal incremental scan would request records after this point.
print("Normal next-window predicate: event_time > watermark")

## 💥 6. Break it: late data

Event `E5` arrives after the watermark but its event time belongs inside an already processed window. A strict `event_time > watermark` filter would miss it.

In [ ]:
print("Late event:")
display(late_batch)
strictly_new = late_batch[late_batch["event_time"] > watermark]
print("Rows accepted by strict watermark filter:", len(strictly_new))
assert len(strictly_new) == 0

## 7. Diagnose → Improve

The watermark is useful for efficiency, but event time can arrive late. Production designs commonly combine a normal watermark with one or more of:

- an overlap/lookback window;
- source sequence numbers;
- change-data-capture positions;
- explicit backfill jobs;
- a destination key that makes replay safe.

Here we use a small lookback and rely on idempotency.

In [ ]:
lookback = watermark - timedelta(minutes=5)
candidate = pd.concat([batch_2, late_batch], ignore_index=True)
candidate = candidate[candidate["event_time"] > lookback]
inserted = load_idempotent(candidate)
print("Lookback candidate rows:", len(candidate))
print("New rows inserted after idempotent deduplication:", inserted)

display(con.execute("SELECT * FROM events ORDER BY event_time").fetchdf())
assert con.execute("SELECT COUNT(*) FROM events").fetchone()[0] == 5

## 8. Partial failure and transaction boundaries

A robust pipeline should not advance its checkpoint independently of the data write. The safest pattern is to make the destination write and checkpoint update part of one transactional unit where the storage system supports it.

In [ ]:
con.execute("BEGIN")
try:
    con.execute("UPDATE checkpoint SET watermark = ? WHERE name='events'", [base + timedelta(minutes=5)])
    con.execute("COMMIT")
except Exception:
    con.execute("ROLLBACK")
    raise
print("Checkpoint advanced only after the transaction committed.")

## 9. What changes in production?

At scale, incremental pipelines need durable checkpoints, source-specific offsets, schema evolution, backfills, retention rules, retry policies and operational metadata. Exactly-once claims must be tied to a concrete source and sink model; “we used a watermark” does not automatically mean exactly-once processing.

**Design question:** What is the stable identity of a record, and what state proves that processing progressed safely?

## 10. Explain it in 60–90 seconds

> Explain why a watermark and idempotency solve different problems. Give one failure where you need both.

## 11. Challenge

Add a `run_id` and a load audit table containing:
- run ID;
- source window;
- input rows;
- inserted rows;
- duplicate rows;
- final watermark;
- status.

Then simulate a failed run and show how an operator can determine whether to retry or backfill.

## Takeaways

- Retries and replays are normal operating conditions.
- Stable record identity enables idempotent writes.
- Watermarks reduce repeated scanning but do not solve late data by themselves.
- Checkpoint advancement must be coordinated with successful data persistence.
- Backfills are part of normal pipeline design, not an emergency afterthought.

---

## Check your work

You have finished the exercise. **Now** compare your reasoning and evidence with the reference solution.

[Open the reference solution for Notebook 13](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/13.md)
